# Domain-tuned retrieval: Colab runner
Runtime → Change runtime type → **T4 GPU**. Run the cells top to bottom.

Phase 1 tunes on **dev**. Phase 3 runs **test** once, for the final numbers.

In [ ]:
# Option A: clone your GitHub repo (replace the URL)
!git clone https://github.com/<your-username>/domain-retrieval-finetune.git
%cd domain-retrieval-finetune

# Option B: upload the zip using the Files panel, then:
# !unzip -q domain-retrieval-finetune.zip && %cd domain-retrieval-finetune

In [ ]:
!pip install -q -r requirements.txt
!python -m pytest -q tests/ || python tests/test_core.py

## Phase 1: baseline and fine-tuning on dev

In [ ]:
!python -m src.evaluate --model sentence-transformers/all-MiniLM-L6-v2 --name minilm-base --split dev

In [ ]:
!python -m src.train --base-model sentence-transformers/all-MiniLM-L6-v2 --output-dir models/minilm-ft --epochs 3

In [ ]:
!python -m src.evaluate --model models/minilm-ft/final --name minilm-ft --split dev

Try `--epochs 1` and `--epochs 5` with different `--output-dir` and `--name` values. Keep whichever scores best **on dev**.

## Phase 2: hard negatives (on dev)

In [ ]:
!python -m src.train --base-model sentence-transformers/all-MiniLM-L6-v2 --output-dir models/minilm-ft-hn --epochs 3 --hard-negatives
!python -m src.evaluate --model models/minilm-ft-hn/final --name minilm-ft-hn --split dev

In [ ]:
!python -m src.make_table --split dev

## Phase 3: final test run (once, with your chosen settings)
Replace `models/minilm-ft/final` below with your best model from the dev table.

In [ ]:
!python -m src.evaluate --model sentence-transformers/all-MiniLM-L6-v2 --name minilm-base --split test
!python -m src.evaluate --model BAAI/bge-small-en-v1.5 --name bge-small-base --split test --query-prefix "Represent this sentence for searching relevant passages: "
!python -m src.evaluate --model models/minilm-ft/final --name minilm-ft --split test --reranker cross-encoder/ms-marco-MiniLM-L-6-v2

In [ ]:
!python -m src.make_table --split test

## Save your work
Colab deletes files when the session ends. Download `results/`, or commit it to GitHub.

In [ ]:
!zip -qr results.zip results models/*/train_config.json
from google.colab import files
files.download('results.zip')

## Optional: publish the model to the Hugging Face Hub

In [ ]:
# from huggingface_hub import notebook_login; notebook_login()
# from sentence_transformers import SentenceTransformer
# SentenceTransformer('models/minilm-ft/final').push_to_hub('<your-hf-username>/minilm-scifact-ft')